# Confronto timeseries cerebellari: Zmin vs Z+ vs SCA6

Confronto dell'attività delle **cellule di Purkinje (PC)** nelle tre condizioni, a livello globale e per regione, più il segnale BOLD (assoluto e normalizzato min-max).

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- DA COMPLETARE: cartella su Drive che contiene i tre file .npz ----
DATA_DIR = "/content/drive/MyDrive/..."

FILES = {
    "Zmin": "cerebellum_zmin_AVG.npz",
    "Z+":   "cerebellum_zplus_AVG.npz",
    "SCA6": "cerebellum_zplus_SCA6_AVG.npz",
}
COLORS = {"Zmin": "purple", "Z+": "green", "SCA6": "blue"}

POP = "PC"        # popolazione da analizzare
N_DISCARD = 100        # punti iniziali da scartare nelle timeseries (transiente)
N_DISCARD_BOLD = 20    # volumi BOLD iniziali da scartare

REGIONS = [
    "Left_I_IV", "Left_V", "Left_VI", "Left_CrusI", "Left_CrusII", "Left_VIIb",
    "Left_VIIIa", "Left_VIIIb", "Left_IX", "Left_X",
    "Vermis_VI", "Vermis_CrusII", "Vermis_VIIb", "Vermis_VIIIa", "Vermis_VIIIb",
    "Vermis_IX", "Vermis_X",
    "Right_I_IV", "Right_V", "Right_VI", "Right_CrusI", "Right_CrusII", "Right_VIIb",
    "Right_VIIIa", "Right_VIIIb", "Right_IX", "Right_X",
]
CONDS = list(FILES)
COMP = ["Z+", "SCA6"]   # condizioni confrontate nelle figure per regione e nel BOLD

## 2. Caricamento dati

In [ ]:
ts, t, bold, bold_t = {}, {}, {}, {}
for cond, fname in FILES.items():
    d = np.load(os.path.join(DATA_DIR, fname))
    ipop = list(d["pop_names"]).index(POP)
    # timeseries: (tempo, popolazione, regione) -> (tempo, regione) per la popolazione scelta
    ts[cond] = d["timeseries_data"][N_DISCARD:, ipop, :]
    t[cond] = d["timeseries_time"][N_DISCARD:] / 1000.0          # s
    bold[cond] = d["bold_data"][N_DISCARD_BOLD:]
    bold_t[cond] = d["bold_time"][N_DISCARD_BOLD:] / 1000.0      # s
    print(f"{cond:5s} {POP}: {ts[cond].shape}   BOLD: {bold[cond].shape}")

assert all(ts[c].shape[1] == len(REGIONS) for c in CONDS)

## 3. Statistiche globali (tutti i campioni tempo x regioni)

In [ ]:
glob = pd.DataFrame(
    {c: {"mean": ts[c].mean(), "std": ts[c].std(), "min": ts[c].min(), "max": ts[c].max()}
     for c in CONDS}
).T
glob.round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, conds in zip(axes, [CONDS, CONDS[1:]]):   # a destra zoom senza Zmin
    ax.bar(conds, glob.loc[conds, "mean"], yerr=glob.loc[conds, "std"],
           color=[COLORS[c] for c in conds], capsize=6)
    ax.set_ylim(bottom=0)
    ax.set_ylabel(f"{POP} firing rate (Hz)")
axes[0].set_title("Media globale ± dev. std")
axes[1].set_title("Zoom: Z+ vs SCA6")
plt.tight_layout()
plt.show()

### Andamento temporale (media sulle regioni)

In [ ]:
plt.figure(figsize=(11, 3.5))
for c in CONDS:
    plt.plot(t[c], ts[c].mean(axis=1), color=COLORS[c], label=c)
plt.xlabel("Tempo (s)")
plt.ylabel(f"{POP} firing rate (Hz)")
plt.legend()
plt.tight_layout()
plt.show()

## 4. Statistiche per regione (nel tempo)

In [ ]:
reg = pd.concat(
    {c: pd.DataFrame({"mean": ts[c].mean(0), "std": ts[c].std(0),
                      "min": ts[c].min(0), "max": ts[c].max(0)}, index=REGIONS)
     for c in CONDS},
    axis=1,
)
reg.round(2)

### 4a. Un pannello per regione (Z+ vs SCA6)

In [ ]:
fig, axes = plt.subplots(4, 7, figsize=(16, 10))
axes = axes.ravel()
for i, r in enumerate(REGIONS):
    ax = axes[i]
    ax.bar(COMP, [reg[c, "mean"][r] for c in COMP], yerr=[reg[c, "std"][r] for c in COMP],
           color=[COLORS[c] for c in COMP], capsize=3)
    ax.set_title(f"{i} {r}", fontsize=10)
    ax.set_ylim(bottom=0)
    ax.tick_params(labelsize=8)
for ax in axes[len(REGIONS):]:
    ax.axis("off")
fig.suptitle(f"{POP} firing rate (Hz): media ± dev. std temporale")
plt.tight_layout()
plt.show()

### 4b. Barre raggruppate

In [ ]:
x = np.arange(len(REGIONS))
w = 0.8 / len(COMP)
plt.figure(figsize=(16, 4.5))
for k, c in enumerate(COMP):
    plt.bar(x + (k - (len(COMP) - 1) / 2) * w, reg[c, "mean"], w, yerr=reg[c, "std"],
            color=COLORS[c], label=c, capsize=2, error_kw={"elinewidth": 0.8})
plt.xticks(x, REGIONS, rotation=60, ha="right")
plt.xlim(-0.6, len(REGIONS) - 0.4)
plt.ylabel(f"{POP} firing rate (Hz)")
plt.legend()
plt.tight_layout()
plt.show()

### 4c. Riduzione percentuale SCA6 vs Z+ per regione

In [ ]:
# riduzione % del firing rate medio: 100 * (Z+ - SCA6) / Z+
red = 100 * (reg["Z+", "mean"] - reg["SCA6", "mean"]) / reg["Z+", "mean"]
red_glob = 100 * (glob.loc["Z+", "mean"] - glob.loc["SCA6", "mean"]) / glob.loc["Z+", "mean"]
print(f"Riduzione globale: {red_glob:.1f}%   |   tra regioni: min {red.min():.1f}%, max {red.max():.1f}%, "
      f"media {red.mean():.1f}%, dev. std {red.std():.1f}%")

plt.figure(figsize=(16, 4.5))
plt.bar(REGIONS, red, color="gray")
plt.axhline(red_glob, color="red", ls="--", label=f"riduzione globale ({red_glob:.1f}%)")
for i, v in enumerate(red):
    plt.text(i, v + 0.5, f"{v:.0f}", ha="center", fontsize=8)
plt.xticks(rotation=60, ha="right")
plt.xlim(-0.6, len(REGIONS) - 0.4)
plt.ylabel(f"Riduzione {POP} firing rate (%)")
plt.legend()
plt.tight_layout()
plt.show()

## 5. BOLD normalizzato

`BOLD = abs(BOLD)`, poi `(BOLD - min) / (max - min)` con min e max globali (su tutte le regioni e tutti i volumi).

In [ ]:
def normalize(x):
    x = np.abs(x)
    return (x - np.min(x)) / (np.max(x) - np.min(x))

bold_norm = {c: normalize(bold[c]) for c in COMP}

# se il BOLD e' identico tra le condizioni, ne plotto una sola
same = all(np.array_equal(bold[COMP[0]], bold[c]) for c in COMP[1:])
print("BOLD identico tra", " e ".join(COMP), ":", same)
bold_conds = COMP[:1] if same else COMP

cmap = plt.get_cmap("turbo")
for c in bold_conds:
    y = bold_norm[c]
    plt.figure(figsize=(12, 4.5))
    for i in range(y.shape[1]):
        plt.plot(bold_t[c], y[:, i], color=cmap(i / (y.shape[1] - 1)), lw=0.8, alpha=0.8)
    plt.plot(bold_t[c], y.mean(1), color="k", lw=2, label="media regioni")
    plt.ylabel("BOLD normalizzato")
    plt.xlabel("Tempo (s)")
    plt.legend(loc="center right")
    plt.title("BOLD normalizzato" + (" (identico in " + " e ".join(COMP) + ")" if same else f" - {c}")
              + "  |  una linea per regione, colore = indice regione")
    plt.tight_layout()
    plt.show()

In [ ]:
# media del BOLD normalizzato per regione
pd.DataFrame({c: bold_norm[c].mean(0) for c in COMP}, index=REGIONS).round(3)